# Step 2: Nighttime-Light Recovery Labels for Multiple Typhoons × All Affected Municipalities

**Goal**: compute one "recovery label" row for every typhoon and every affected municipality: whether an outage was detected, how many weeks it took to return to 80% of the pre-event level, and data-quality flags. This table is what Steps 3–4 will try to predict.

**What changes from Step 1**
1. Unit of analysis: from 3 km circles around city centres to **all municipal boundaries in the Philippines** (geoBoundaries ADM3)
2. Only **built-up** pixels (GHSL 2020) that were actually lit before the event are counted, so dark background is never mistaken for an outage
3. **Bright-moon nights** (lunar illumination > 80%) are dropped, reducing false signals from moonlit clouds
4. **Date alignment**: Black Marble dates are UTC days; the image dated D is acquired around 01:30 local time on D+1 in the Philippines. This notebook compares that acquisition time with the landfall time
5. **Back-to-back storms**: if a municipality is hit again by a later storm on the list within the observation window, the earlier storm's record stops at the later landfall (censored). For earlier storms that are *not* on the list, the baseline is ended manually before them
6. Computation runs on Google's servers and the results are exported to your Google Drive

**Expected time**: about 10 minutes of setup; export tasks run in parallel on the server, roughly 10–40 minutes per typhoon. You can close Colab while they run and resume as described in cell 9.

**Before running**: replace `PROJECT_ID` in cell 2 with your own project ID.

In [ ]:
# 1. Install dependencies
!pip install -q earthengine-api geemap geopandas

In [ ]:
# 2. Authenticate Earth Engine and mount Google Drive (exports are saved there)
import os
import ee

PROJECT_ID = "ee-yourname"   # replace with your own Cloud project ID
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

from google.colab import drive
drive.mount("/content/drive")
DRIVE_FOLDER = "ph_typhoon_ntl"
DRIVE_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs("figures", exist_ok=True)
print("OK")

## Typhoon list

Landfall times are Philippine local time (PHT), taken from PAGASA bulletins and major news reports; cell 4 cross-checks them against IBTrACS best tracks.

- **core**: main modelling sample, all after 2017 so AlphaEarth satellite embeddings are available
- **reference**: Haiyan, kept only as the motivating case, not used for modelling
- **optional**: sparsely lit landfall areas (Catanduanes, Aurora) or crowded storm sequences, so label quality may be lower

`baseline_end` ends the baseline early to avoid earlier disasters that are not on the list:
- Phanfone: avoids Kammuri in early December
- Goni and Vamco: avoid Molave in late October
- Man-yi: avoids the run of storms from late October (Trami and others)
- Kalmaegi and Fung-wong: avoid the late-September 2025 storms and the northern Cebu earthquake

In [ ]:
# 3. Typhoon list and parameters
import numpy as np
import pandas as pd

EVENTS = pd.DataFrame([
    # key,             IBTrACS name, season, PAGASA name, first PH landfall (PHT), place,                          baseline_end, role
    ("haiyan_2013",    "HAIYAN",     2013, "Yolanda", "2013-11-08 04:40", "Guiuan, Eastern Samar",        None,         "reference"),
    ("mangkhut_2018",  "MANGKHUT",   2018, "Ompong",  "2018-09-15 01:40", "Baggao, Cagayan",              None,         "core"),
    ("kammuri_2019",   "KAMMURI",    2019, "Tisoy",   "2019-12-02 23:00", "Gubat, Sorsogon",              None,         "core"),
    ("phanfone_2019",  "PHANFONE",   2019, "Ursula",  "2019-12-24 16:45", "Salcedo, Eastern Samar",       "2019-11-28", "core"),
    ("goni_2020",      "GONI",       2020, "Rolly",   "2020-11-01 04:50", "Bato, Catanduanes",            "2020-10-20", "core"),
    ("vamco_2020",     "VAMCO",      2020, "Ulysses", "2020-11-11 22:30", "Patnanungan, Quezon",          "2020-10-20", "core"),
    ("rai_2021",       "RAI",        2021, "Odette",  "2021-12-16 13:30", "Siargao, Surigao del Norte",   None,         "core"),
    ("noru_2022",      "NORU",       2022, "Karding", "2022-09-25 17:30", "Burdeos, Quezon",              None,         "core"),
    ("manyi_2024",     "MAN-YI",     2024, "Pepito",  "2024-11-16 21:40", "Panganiban, Catanduanes",      "2024-10-18", "optional"),
    ("kalmaegi_2025",  "KALMAEGI",   2025, "Tino",    "2025-11-04 00:00", "Silago, Southern Leyte",       "2025-09-20", "core"),
    ("fungwong_2025",  "FUNG-WONG",  2025, "Uwan",    "2025-11-09 21:10", "Dinalungan, Aurora",           "2025-09-20", "optional"),
], columns=["key", "ibtracs_name", "season", "local_name", "landfall", "landfall_place", "baseline_end", "role"])
EVENTS["landfall"] = pd.to_datetime(EVENTS["landfall"])

RUN_ROLES = ["reference", "core"]   # add "optional" to include the optional events
RUN = EVENTS[EVENTS.role.isin(RUN_ROLES)].reset_index(drop=True)

# Parameters
BASE_DAYS = 120          # baseline length (days)
BASE_GAP_DAYS = 3        # baseline ends 3 days before landfall (storm clouds, pre-emptive shutdowns)
POST_WEEKS = 26          # follow each event for 26 weeks
SWATH_KM = 150           # municipalities within 150 km of the track
BUILT_MIN_FRAC = 0.03    # at least 3% built-up surface in a 500 m pixel
MIN_RADIANCE = 1.0       # pre-event median radiance of at least 1 nW/cm²/sr
MIN_ELIGIBLE_PX = 8      # at least 8 eligible pixels (~2 km²) per municipality
MIN_VALID_FRAC = 0.5     # at least half of eligible pixels clear on a given night
MOON_MAX_ILLUM = 0.8     # drop nights with lunar illumination > 80%
MIN_DAYS_PER_WEEK = 2    # at least 2 valid nights per week
MIN_BASE_NIGHTS = 8      # flag low_baseline if fewer than 8 baseline nights
OUTAGE_LEVEL = 80        # outage = weeks 0–3 minimum below 80% (and beyond noise)
RECOVERY_LEVEL = 80      # recovered = two consecutive available weeks ≥ 80%

# The image dated D (UTC day) is acquired around 01:30 PHT on D+1
OVERPASS_LAG = pd.Timedelta(hours=25, minutes=30)

def windows(ev):
    # returns baseline start/end and follow-up end (local time), plus the image-date range to request
    if isinstance(ev.baseline_end, str):
        base_end = pd.Timestamp(ev.baseline_end)
    else:
        base_end = ev.landfall - pd.Timedelta(days=BASE_GAP_DAYS)
    base_start = base_end - pd.Timedelta(days=BASE_DAYS)
    post_end = ev.landfall + pd.Timedelta(weeks=POST_WEEKS)
    img_start = (base_start - OVERPASS_LAG).normalize() - pd.Timedelta(days=1)
    img_end = (post_end - OVERPASS_LAG).normalize() + pd.Timedelta(days=2)
    return base_start, base_end, post_end, img_start, img_end

RUN[["key", "local_name", "landfall", "landfall_place", "baseline_end", "role"]]

In [ ]:
# 4. Load IBTrACS tracks and cross-check the listed landfall times
IBTRACS_URL = ("https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/"
               "v04r01/access/csv/ibtracs.WP.list.v04r01.csv")
cols = ["SID", "SEASON", "NAME", "ISO_TIME", "LAT", "LON", "USA_WIND", "WMO_WIND", "DIST2LAND"]
ib = pd.read_csv(IBTRACS_URL, usecols=cols, skiprows=[1], low_memory=False)   # row 2 holds units
ib["SEASON"] = pd.to_numeric(ib["SEASON"], errors="coerce")
ib = ib[ib["SEASON"] >= 2013].copy()
for c in ["LAT", "LON", "USA_WIND", "WMO_WIND", "DIST2LAND"]:
    ib[c] = pd.to_numeric(ib[c], errors="coerce")
ib["time_pht"] = pd.to_datetime(ib["ISO_TIME"]) + pd.Timedelta(hours=8)

PH_LON, PH_LAT = (114, 130), (3, 22)
tracks, checks = {}, []
for ev in RUN.itertuples():
    t = ib[(ib.NAME == ev.ibtracs_name) & (ib.SEASON == ev.season)]
    if t.SID.nunique() > 1:
        gap = (t.time_pht - ev.landfall).abs()
        t = t[t.SID == t.assign(gap=gap).groupby("SID").gap.min().idxmin()]
    t = t[t.LON.between(*PH_LON) & t.LAT.between(*PH_LAT)].sort_values("time_pht")
    assert len(t) > 1, f"{ev.key}: no IBTrACS track found"
    tracks[ev.key] = t
    near = t[(t.time_pht - ev.landfall).abs() <= pd.Timedelta(days=2)]
    land = near[near.DIST2LAND <= 0]
    first_land = land.time_pht.min() if len(land) else pd.NaT
    checks.append({
        "event": ev.key,
        "listed landfall (PHT)": ev.landfall,
        "IBTrACS first point over land (PHT)": first_land,
        "hours apart": round((first_land - ev.landfall) / pd.Timedelta(hours=1), 1) if pd.notna(first_land) else None,
        "max 1-min wind within ±2 days (kt)": near.USA_WIND.max(),
    })
# IBTrACS is 3-hourly, so gaps up to ~3 h are normal; check any gap above 12 h
pd.DataFrame(checks)

In [ ]:
# 5. Municipal boundaries (geoBoundaries ADM3), matched to provinces (ADM2)
import requests
import geopandas as gpd

def load_gb(level):
    try:
        meta = requests.get(f"https://www.geoboundaries.org/api/current/gbOpen/PHL/{level}/", timeout=60).json()
        return gpd.read_file(meta["simplifiedGeometryGeoJSON"])
    except Exception as e:
        print(f"{level}: API failed, trying the GitHub mirror ({e})")
        return gpd.read_file("https://media.githubusercontent.com/media/wmgeolab/geoBoundaries/main/releaseData/"
                             f"gbOpen/PHL/{level}/geoBoundaries-PHL-{level}_simplified.geojson")

adm3 = load_gb("ADM3").to_crs(4326)[["shapeID", "shapeName", "geometry"]].rename(columns={"shapeName": "muni"})
adm2 = load_gb("ADM2").to_crs(4326)[["shapeName", "geometry"]].rename(columns={"shapeName": "province"})
pts = adm3.set_geometry(adm3.representative_point())
prov = gpd.sjoin(pts, adm2, predicate="within", how="left")[["shapeID", "province"]].drop_duplicates("shapeID")
adm3 = adm3.merge(prov, on="shapeID", how="left")
print(f"{len(adm3)} municipalities/cities; {adm3.province.notna().mean():.0%} matched to a province")

In [ ]:
# 6. Select municipalities by distance to the track; flag those interrupted by a later storm
from shapely.geometry import LineString
import matplotlib.pyplot as plt

UTM = 32651
anchor = adm3.to_crs(UTM).representative_point()
swaths = {}
for ev in RUN.itertuples():
    t = tracks[ev.key]
    line = gpd.GeoSeries([LineString(zip(t.LON, t.LAT))], crs=4326).to_crs(UTM).iloc[0]
    d = anchor.distance(line).values / 1000
    sel = adm3[d <= SWATH_KM].copy()
    sel["dist_km"] = d[d <= SWATH_KM].round(1)
    swaths[ev.key] = sel

# Back-to-back storms: stop the earlier record at the later landfall for municipalities hit by both
censor = {}
for a in RUN.itertuples():
    for b in RUN.itertuples():
        if pd.Timedelta(0) < b.landfall - a.landfall < pd.Timedelta(weeks=POST_WEEKS):
            for sid in set(swaths[a.key].shapeID) & set(swaths[b.key].shapeID):
                censor[(a.key, sid)] = min(censor.get((a.key, sid), b.landfall), b.landfall)

print(pd.DataFrame([{"event": k, "municipalities in swath": len(v),
                     "cut short by a later storm": sum(1 for s in v.shapeID if (k, s) in censor)}
                    for k, v in swaths.items()]).to_string(index=False))

INK, INK2, SURFACE = "#1f1f1e", "#5f5e5a", "#fcfcfb"
plt.rcParams.update({"font.size": 10, "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2,
                     "ytick.color": INK2, "axes.edgecolor": "#c9c8c4", "figure.facecolor": SURFACE,
                     "axes.facecolor": SURFACE})
fig, ax = plt.subplots(figsize=(7, 8.5), constrained_layout=True)
adm2.plot(ax=ax, color="#e8e6e1", edgecolor="white", linewidth=0.3)
for ev in RUN.itertuples():
    t = tracks[ev.key]
    ax.plot(t.LON, t.LAT, color=INK2, lw=1.2)
    i = int(np.argmin(np.abs((t.time_pht - ev.landfall) / pd.Timedelta(hours=1))))
    ax.annotate(f"{ev.local_name} {ev.season}", (t.LON.iloc[i], t.LAT.iloc[i]), xytext=(4, 4),
                textcoords="offset points", fontsize=8, color=INK)
ax.set_xlim(116, 128); ax.set_ylim(4.5, 21.5); ax.set_aspect("equal")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("Typhoon tracks in the study (IBTrACS)", loc="left", fontsize=11)
fig.savefig("figures/step2_tracks.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# 7. Earth Engine: nighttime lights (quality + cloud screening) and built-up mask
import geemap

COLLECTION = "NASA/VIIRS/002/VNP46A2"
NTL_BAND, QF_BAND, CLOUD_BAND = "DNB_BRDF_Corrected_NTL", "Mandatory_Quality_Flag", "QF_Cloud_Mask"
raw_col = ee.ImageCollection(COLLECTION)
proj = raw_col.first().select(NTL_BAND).projection()

def clean(img):
    ok = img.select(QF_BAND).lte(1)                                  # 0/1 = high quality
    cloud = img.select(CLOUD_BAND).rightShift(6).bitwiseAnd(3)        # bits 6–7: 0/1 clear, 2/3 cloudy
    return (img.select(NTL_BAND).updateMask(ok.And(cloud.lte(1))).rename("ntl")
               .copyProperties(img, ["system:time_start"]))

ntl = raw_col.map(clean)

# GHSL 2020 built-up surface (m² per 100 m cell) → built-up fraction of each 500 m pixel
built = (ee.ImageCollection("JRC/GHSL/P2023A/GHS_BUILT_S").filterDate("2020-01-01", "2021-01-01").first()
           .select("built_surface").divide(10000)
           .reduceResolution(ee.Reducer.mean(), maxPixels=64).reproject(proj))
settlement = built.gte(BUILT_MIN_FRAC)

latest = ee.Date(raw_col.aggregate_max("system:time_start")).format("YYYY-MM-dd").getInfo()
print("Latest VNP46A2 image:", latest)
for ev in RUN.itertuples():
    if windows(ev)[4] > pd.Timestamp(latest):
        print(f"  WARNING: {ev.key}: the 26-week window extends past the latest image; recovery will be censored there")

## Export (runs on Google's servers)

Cell 8 starts one export task per typhoon: it first counts eligible pixels in each municipality, then extracts the mean radiance of every night in the observation window into the `ph_typhoon_ntl` folder in your Drive.

- Tasks run on the server and **keep going if you close Colab**. Track them at <https://code.earthengine.google.com/tasks>
- If you close Colab midway: rerun cells 1–7, then cell 8. Finished or still-running tasks are skipped automatically; then continue from cell 9

In [ ]:
# 8. Start one export task per typhoon
import time

REDUCER = ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True)

def active_tasks():
    try:
        return {t["description"] for t in ee.data.getTaskList() if t["state"] in ("READY", "RUNNING")}
    except Exception:
        return set()

running = active_tasks()
tasks = {}
for ev in RUN.itertuples():
    base_start, base_end, post_end, img_start, img_end = windows(ev)
    sel = swaths[ev.key]
    fc = geemap.gdf_to_ee(sel[["shapeID", "geometry"]])

    # eligible pixels: built-up and pre-event median radiance ≥ MIN_RADIANCE
    base = (ntl.filterDate((base_start - OVERPASS_LAG).strftime("%Y-%m-%d"),
                           (base_end - OVERPASS_LAG).strftime("%Y-%m-%d"))
               .median().reproject(proj))
    eligible = settlement.And(base.gte(MIN_RADIANCE)).reproject(proj)
    counts = eligible.selfMask().reduceRegions(fc, ee.Reducer.count(), crs=proj).getInfo()
    elig = pd.DataFrame([{"shapeID": f["properties"]["shapeID"],
                          "eligible_px": f["properties"].get("count") or 0} for f in counts["features"]])
    meta = sel.drop(columns="geometry").merge(elig, on="shapeID", how="left")
    meta["censor_at"] = [censor.get((ev.key, s)) for s in meta.shapeID]
    meta.to_csv(f"{DRIVE_DIR}/meta_{ev.key}.csv", index=False)
    keep = meta.loc[meta.eligible_px >= MIN_ELIGIBLE_PX, "shapeID"].tolist()
    msg = f"{ev.key}: {len(sel)} in swath, {len(keep)} with enough lit pixels"

    desc = f"ntl_{ev.key}"
    if os.path.exists(f"{DRIVE_DIR}/{desc}.csv") or desc in running:
        print(msg, "-> already exported or running, skipped")
        continue

    fc_keep = fc.filter(ee.Filter.inList("shapeID", keep))
    def per_image(img, eligible=eligible, fc_keep=fc_keep):
        date = img.date().format("YYYY-MM-dd")
        return (img.updateMask(eligible).reduceRegions(fc_keep, REDUCER, crs=proj)
                   .map(lambda f: f.set("date", date)))
    daily = (ntl.filterDate(img_start.strftime("%Y-%m-%d"), img_end.strftime("%Y-%m-%d"))
                .map(per_image).flatten())
    task = ee.batch.Export.table.toDrive(collection=daily, description=desc, folder=DRIVE_FOLDER,
                                         fileNamePrefix=desc, fileFormat="CSV",
                                         selectors=["shapeID", "date", "mean", "count"])
    task.start()
    tasks[ev.key] = task
    print(msg, "-> export started")

In [ ]:
# 9. Wait for the exports (refreshes every minute). If you closed Colab, you can skip this cell and check the Tasks page instead
def wait_for(tasks, every=60):
    while tasks:
        states = {k: t.status()["state"] for k, t in tasks.items()}
        print(time.strftime("%H:%M:%S"), states)
        if all(s in ("COMPLETED", "FAILED", "CANCELLED") for s in states.values()):
            break
        time.sleep(every)
    for k, t in tasks.items():
        if t.status()["state"] == "FAILED":
            print(f"{k} FAILED:", t.status().get("error_message"))

wait_for(tasks)

## Compute recovery labels

For every municipality and typhoon:
1. Keep valid nights only: at least half of the eligible pixels clear, lunar illumination ≤ 80%
2. Baseline = median radiance over valid nights in the baseline window; noise = standard deviation of weekly medians in that window
3. After landfall, take weekly medians (at least 2 nights per week)
4. **Outage**: the minimum over weeks 0–3 is below 80% and below 100% − 2 × noise
5. **Weeks to recovery**: after the minimum, the first time two consecutive available weeks are ≥ 80%; if that never happens within the window, the record is censored (`censored = True`)

What the `status` column means:
- `ok`: normal
- `low_baseline`: fewer than 8 baseline nights; usable with caution
- `no_early_data`: weeks 0–3 fully clouded out, so no call can be made
- `no_baseline`: fewer than 3 baseline nights
- `too_few_lit_pixels`: too little light to measure. These are mostly rural municipalities, and the equity analysis in notebook 04 treats them separately

> If a file is reported missing: Drive can take a minute or two to sync; wait and rerun the cell.

In [ ]:
# 10. Read the exports and compute a recovery label for each municipality
import glob

REF_NEW_MOON = pd.Timestamp("2000-01-06 18:14")    # UTC reference new moon for mean lunar phase
SYNODIC = 29.530588853

def moon_illum(obs_utc):
    age = ((obs_utc - REF_NEW_MOON) / pd.Timedelta(days=1)) % SYNODIC
    return (1 - np.cos(2 * np.pi * age / SYNODIC)) / 2

def load_event(key):
    files = sorted(glob.glob(f"{DRIVE_DIR}/ntl_{key}*.csv"))
    if not files:
        return None
    d = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    d["date"] = pd.to_datetime(d["date"])
    d["obs_pht"] = d["date"] + OVERPASS_LAG
    d["illum"] = moon_illum(d["obs_pht"] - pd.Timedelta(hours=8))
    return d

def week_of(obs, landfall):
    # days (0, 7] after landfall = week 0; (-7, 0] before = week -1
    return (np.ceil((obs - landfall) / pd.Timedelta(days=1) / 7) - 1).astype(int)

def label_muni(g, landfall, base_start, base_end, post_end, censor_at):
    ok = g["mean"].notna() & (g["count"] >= MIN_VALID_FRAC * g["eligible_px"]) & (g["illum"] <= MOON_MAX_ILLUM)
    g = g[ok]
    base = g[(g.obs_pht >= base_start) & (g.obs_pht <= base_end)]
    out = {"base_nights": len(base)}
    if len(base) < 3:
        return {**out, "status": "no_baseline"}
    b = base["mean"].median()
    g = g.assign(pct=g["mean"] / b * 100, wk=week_of(g.obs_pht, landfall))

    pre = g.loc[base.index].groupby("wk").pct.agg(["median", "size"])
    noise = pre.loc[pre["size"] >= MIN_DAYS_PER_WEEK, "median"].std()

    end = min(post_end, censor_at) if pd.notna(censor_at) else post_end
    post = g[(g.obs_pht > landfall) & (g.obs_pht < end)]
    w = post.groupby("wk").pct.agg(["median", "size"])
    w = w.loc[w["size"] >= MIN_DAYS_PER_WEEK, "median"]
    out.update(baseline=round(b, 3), noise=round(noise, 1) if pd.notna(noise) else np.nan, post_weeks=len(w))
    for k in (1, 2, 4, 8, 12):
        near = w[(w.index >= k - 1) & (w.index <= k + 1)]
        out[f"pct_w{k}"] = round(near.iloc[np.argmin(np.abs(near.index - k))]) if len(near) else np.nan

    early = w[w.index <= 3]
    if early.empty:
        return {**out, "status": "no_early_data"}
    low, low_wk = early.min(), early.idxmin()
    threshold = OUTAGE_LEVEL if pd.isna(noise) else min(OUTAGE_LEVEL, 100 - 2 * noise)
    status = "ok" if len(base) >= MIN_BASE_NIGHTS else "low_baseline"
    out.update(min_pct_w0_3=round(low), outage=bool(low < threshold), status=status)
    if not out["outage"]:
        return {**out, "recovery_weeks": 0, "censored": False}

    after = w[w.index >= low_wk]
    hit = [k for k, now, nxt in zip(after.index[:-1], after.values[:-1], after.values[1:])
           if now >= RECOVERY_LEVEL and nxt >= RECOVERY_LEVEL]
    if hit:
        return {**out, "recovery_weeks": int(hit[0]) + 1, "censored": False}
    return {**out, "recovery_weeks": int(w.index.max()) + 1, "censored": True}

rows = []
for ev in RUN.itertuples():
    d = load_event(ev.key)
    if d is None:
        print(f"{ev.key}: no export file yet")
        continue
    meta = pd.read_csv(f"{DRIVE_DIR}/meta_{ev.key}.csv", parse_dates=["censor_at"])
    d = d.merge(meta[["shapeID", "eligible_px"]], on="shapeID", how="left")
    base_start, base_end, post_end, _, _ = windows(ev)
    info = meta.set_index("shapeID")
    for sid, g in d.groupby("shapeID"):
        m = info.loc[sid]
        r = label_muni(g, ev.landfall, base_start, base_end, post_end, m.censor_at)
        rows.append({"event": ev.key, "shapeID": sid, "muni": m.muni, "province": m.province,
                     "dist_km": m.dist_km, "eligible_px": m.eligible_px, "censor_at": m.censor_at, **r})
    for m in meta[meta.eligible_px.fillna(0) < MIN_ELIGIBLE_PX].itertuples():
        rows.append({"event": ev.key, "shapeID": m.shapeID, "muni": m.muni, "province": m.province,
                     "dist_km": m.dist_km, "eligible_px": m.eligible_px, "status": "too_few_lit_pixels"})

labels = pd.DataFrame(rows)
labels.groupby("event", sort=False).status.value_counts().unstack(fill_value=0)

In [ ]:
# 11. Summary by typhoon
def summarize(x):
    ok = x[x.status.isin(["ok", "low_baseline"])]
    out = ok[ok.outage.astype(bool)]
    return pd.Series({
        "in swath": len(x),
        "too few lit pixels": int((x.status == "too_few_lit_pixels").sum()),
        "labelled": len(ok),
        "outage detected": len(out),
        "median weeks to 80% (recovered only)": out.loc[~out.censored.astype(bool), "recovery_weeks"].median(),
        "not recovered in window": int(out.censored.astype(bool).sum()),
    })

summary = labels.groupby("event", sort=False).apply(summarize, include_groups=False)
int_cols = [c for c in summary.columns if not c.startswith("median")]
summary[int_cols] = summary[int_cols].astype(int)
summary

## Figures

- **Maps**: one per typhoon, municipalities coloured by weeks to recovery. Hatching = no call possible (no data in weeks 0–3, or interrupted by the next storm before recovering)
- **Distance scatter**: the closer to the track, the lower the weeks 0–3 minimum should be. If that gradient is missing, the labels may have a problem

In [ ]:
# 12. Maps of weeks to recovery (small multiples)
from matplotlib.patches import Patch

CATS = [("No outage detected", "#cfcdc6"), ("≤ 2 weeks", "#86b6ef"), ("3–4 weeks", "#5598e7"),
        ("5–8 weeks", "#2a78d6"), ("9–16 weeks", "#1c5cab"), ("> 16 weeks or not yet", "#0d366b")]

def category(r):
    if r.status not in ("ok", "low_baseline"):
        return None
    if not r.outage:
        return 0
    w = r.recovery_weeks
    if r.censored:
        return 5 if w > 16 else None
    return 1 if w <= 2 else 2 if w <= 4 else 3 if w <= 8 else 4 if w <= 16 else 5

keys = [k for k in RUN.key if k in set(labels.event)]
ncol = 3
nrow = int(np.ceil(len(keys) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 5.2 * nrow), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()
for ax, key in zip(axes, keys):
    ev = RUN.set_index("key").loc[key]
    lab = labels[labels.event == key].copy()
    lab["cat"] = lab.apply(category, axis=1)
    geo = adm3.merge(lab[["shapeID", "cat", "status"]], on="shapeID")
    x0, y0, x1, y1 = geo.total_bounds
    pad = 0.3
    adm3.cx[x0 - pad:x1 + pad, y0 - pad:y1 + pad].plot(ax=ax, color="#f1f0ec", edgecolor="white", linewidth=0.2)
    unknown = geo[geo.cat.isna() & (geo.status != "too_few_lit_pixels")]
    if len(unknown):
        unknown.plot(ax=ax, facecolor="white", edgecolor="#a3a29d", hatch="////", linewidth=0.2)
    for i, (_, color) in enumerate(CATS):
        part = geo[geo.cat == i]
        if len(part):
            part.plot(ax=ax, color=color, edgecolor="white", linewidth=0.2)
    t = tracks[key]
    ax.plot(t.LON, t.LAT, color=INK, lw=1.2)
    ax.set_xlim(x0 - pad, x1 + pad); ax.set_ylim(y0 - pad, y1 + pad); ax.set_aspect("equal")
    ax.set_title(f"{ev.local_name} ({ev.ibtracs_name.title()}), {ev.season}", loc="left", fontsize=11)
    ax.tick_params(labelsize=8)
    ax.set_xlabel(""); ax.set_ylabel("")
for ax in axes[len(keys):]:
    ax.axis("off")
handles = [Patch(color=c, label=l) for l, c in CATS]
handles += [Patch(facecolor="white", edgecolor="#a3a29d", hatch="////", label="No call possible"),
            Patch(color="#f1f0ec", label="Too little light / outside swath")]
fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.05))
fig.suptitle("Weeks for municipal nighttime lights to return to 80% of pre-event levels · NASA Black Marble",
             x=0.01, ha="left", fontsize=13)
fig.savefig("figures/step2_recovery_maps.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# 13. Sanity check: weeks 0–3 minimum vs distance to the track
ok = labels[labels.status.isin(["ok", "low_baseline"])]
fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 3.4 * nrow), sharex=True, sharey=True,
                         constrained_layout=True)
axes = np.atleast_1d(axes).ravel()
bins = np.arange(0, SWATH_KM + 25, 25)
for ax, key in zip(axes, keys):
    ev = RUN.set_index("key").loc[key]
    x = ok[ok.event == key]
    ax.scatter(x.dist_km, x.min_pct_w0_3.clip(0, 150), s=10, color="#2a78d6", alpha=0.45, linewidths=0)
    med = x.groupby(pd.cut(x.dist_km, bins), observed=True).min_pct_w0_3.median()
    ax.plot([b.mid for b in med.index], med.values, color="#0d366b", lw=2, marker="o", ms=4)
    ax.axhline(OUTAGE_LEVEL, color="#a3a29d", lw=1, ls=":")
    ax.set_title(f"{ev.local_name} {ev.season} (n = {len(x)})", loc="left", fontsize=10)
    ax.grid(axis="y", color="#e5e4e0", lw=0.8)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
for ax in axes[len(keys):]:
    ax.axis("off")
fig.supxlabel("Distance from municipality to typhoon track (km)", color=INK2)
fig.supylabel("Weeks 0–3 minimum (% of baseline)", color=INK2)
fig.suptitle("Closer to the track, darker afterwards? Dots = municipalities; line = median per 25 km",
             x=0.01, ha="left", fontsize=12)
fig.savefig("figures/step2_distance_check.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# 14. Save and download (utf-8-sig so Excel opens it cleanly)
labels.to_csv("figures/step2_labels.csv", index=False, encoding="utf-8-sig")
labels.to_csv(f"{DRIVE_DIR}/step2_labels.csv", index=False, encoding="utf-8-sig")
summary.to_csv("figures/step2_summary.csv", encoding="utf-8-sig")
from google.colab import files
for f in ["step2_summary.csv", "step2_labels.csv", "step2_recovery_maps.png", "step2_distance_check.png"]:
    files.download(f"figures/{f}")